<img src="https://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Synthetic Financial Time Series

**Sequential GAN: LSTM Generator with Per-Step Noise and CNN Discriminator (PyTorch)**

&copy; Dr. Yves J. Hilpisch

<a href="https://tpq.io" target="_blank">https://tpq.io</a> | <a href="https://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:team@tpq.io">team@tpq.io</a>

## Imports & Settings

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pylab import plt, mpl

In [ ]:
import torch

In [ ]:
sys.path.insert(0, os.path.abspath('..'))
from synthts.data import load_eod, compute_log_returns, ReturnScaler
from synthts.sequential import (SeqGenerator, SeqDiscriminator,
    train_seq_gan, generate_sequences)
from synthts.evaluate import (ks_test, pass_rate, plot_histograms,
    plot_sorted, plot_pvalue_histogram, sample_windows,
    sequence_ks_pvalues, compare_acf_windows,
    plot_acf_window_comparison, plot_drawdown_window_comparison)

In [ ]:
plt.style.use('seaborn-v0_8')
mpl.rcParams['font.family'] = 'serif'

## Data Preparation

In [ ]:
prices = load_eod(symbol='GLD')
rets = compute_log_returns(prices)
scaler = ReturnScaler()
rets_ = scaler.fit_transform(rets)

In [ ]:
f'Returns: {len(rets)} observations'

## Model Training

The sequential GAN uses an LSTM generator with **per-step noise**
(global latent + independent noise at each time step) and a
**1-D CNN discriminator** with spectral normalisation. Training
uses one-sided label smoothing, multiple generator steps, and
an auxiliary statistic loss for marginal moments, quantiles,
and short-lag autocorrelations.

In [ ]:
SEQ_LEN = 63  # 3-month window
LATENT_DIM = 8
NOISE_DIM = 4
HU_GEN = 64
HU_DISC = 64

In [ ]:
torch.manual_seed(100)
np.random.seed(100)
g = SeqGenerator(latent_dim=LATENT_DIM, noise_dim=NOISE_DIM,
                 hidden_units=HU_GEN, output_dim=1,
                 seq_len=SEQ_LEN, num_layers=1)
d = SeqDiscriminator(input_dim=1, hidden_units=HU_DISC)

In [ ]:
%%time
history = train_seq_gan(
    g, d, rets_.reshape(-1, 1),
    seq_len=SEQ_LEN, epochs=1501, batch_size=128,
    lr_d=0.0001, lr_g=0.0002, latent_dim=LATENT_DIM,
    label_smooth=0.1, n_gen_steps=2,
    stat_weight=5.0, stat_max_lag=5,
    seed=100, log_interval=500
)

## Generating Synthetic Sequences

In [ ]:
N = 100
seqs = generate_sequences(g, N, latent_dim=LATENT_DIM, seed=101)
seqs.shape

In [ ]:
synth_windows = np.array([
    scaler.inverse_transform(seqs[i, :, 0])
    for i in range(N)
])
real_windows = sample_windows(rets, SEQ_LEN, N, rng_seed=100)
synth_flat = synth_windows.flatten()
real_flat = real_windows.flatten()
f'Window observations: {len(real_flat)} real, {len(synth_flat)} synthetic'

### Marginal Distribution

In [ ]:
plot_histograms(real_flat, synth_flat);

In [ ]:
plot_sorted(real_flat, synth_flat);

In [ ]:
print(f'Mean  | Real: {real_flat.mean():.6f}  Synth: {synth_flat.mean():.6f}')
print(f'Std   | Real: {real_flat.std():.6f}  Synth: {synth_flat.std():.6f}')
print(f'Min   | Real: {real_flat.min():.6f}  Synth: {synth_flat.min():.6f}')
print(f'Max   | Real: {real_flat.max():.6f}  Synth: {synth_flat.max():.6f}')

In [ ]:
ks_result = ks_test(real_flat, synth_flat)
f'Flattened window KS statistic: {ks_result["statistic"]:.4f}'

### Window KS Test

In [ ]:
pvals = sequence_ks_pvalues(real_windows, synth_windows)
f'Window KS pass rate: {pass_rate(pvals):.0%}'

In [ ]:
plot_pvalue_histogram(pvals);

### Window Autocorrelation Comparison

In [ ]:
comp = compare_acf_windows(real_windows, synth_windows, max_lag=10)
f'ACF diff: {comp["acf_diff"]:.4f}, Sq-return ACF diff: {comp["acf2_diff"]:.4f}'

In [ ]:
plot_acf_window_comparison(real_windows, synth_windows, max_lag=10);

### Window Cumulative Returns

In [ ]:
n_show = min(10, N)
fig, ax = plt.subplots()
for i in range(n_show):
    ax.plot(np.arange(SEQ_LEN), np.cumsum(real_windows[i]),
            'r', lw=0.9, alpha=0.7, label='_nolegend_')
    ax.plot(np.arange(SEQ_LEN), np.cumsum(synth_windows[i]),
            'b--', lw=0.7, alpha=0.8, label='_nolegend_')
ax.plot([], [], 'r', lw=0.9, label='real')
ax.plot([], [], 'b--', lw=0.7, label='synthetic')
ax.legend();
ax.set_title('Cumulative Returns');
ax.set_xlabel('Day');
ax.set_ylabel('Cumulative Return');

### Window Drawdown Comparison

In [ ]:
plot_drawdown_window_comparison(real_windows, synth_windows);

## Summary

The sequential GAN with an LSTM generator (per-step noise)
and a 1-D CNN discriminator with spectral normalisation
produces stable training dynamics.  The generator produces
return sequences with realistic marginal statistics
(mean, std) and reasonable temporal structure.  The approach
is more challenging than the dense (vanilla/WGAN-GP) models
because the discriminator can exploit temporal patterns,
but the CNN architecture provides a good balance between
discriminative power and training stability.

In [ ]:
print(f'Mean  | Real: {real_flat.mean():.6f}  Synth: {synth_flat.mean():.6f}')
print(f'Std   | Real: {real_flat.std():.6f}  Synth: {synth_flat.std():.6f}')
print(f'KS stat | {ks_result["statistic"]:.4f}')
print(f'KS pass rate | {pass_rate(pvals):.0%}')
print(f'ACF   | diff={comp["acf_diff"]:.4f}  sq-ret diff={comp["acf2_diff"]:.4f}')

<img src="https://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="https://tpq.io" target="_blank">https://tpq.io</a> | <a href="https://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:team@tpq.io">team@tpq.io</a>